# Rolling IPCA forecast accuracy: 3, 6, and 12 months

**Research questions:** How do Focus-based forecast errors differ across rolling 3-, 6-, and 12-month horizons? Does disagreement in the BCB's published 12-month forecasts signal larger eventual errors?

All three constructed forecasts compound the same monthly Focus medians, sampled at the same **279 month-end dates (November 2001–January 2025)**. The BCB's published, non-smoothed 12-month median is retained separately as a benchmark and as the source of 12-month forecaster dispersion. A compounded sequence of monthly medians is **not** the median of institution-level compounded forecasts. The latter are unavailable here. Sources: [BCB Focus data](https://dadosabertos.bcb.gov.br/dataset/expectativas-mercado), [BCB 12-month methodology](https://www.bcb.gov.br/content/controleinflacao/Documents/Expectativas_mercado/FAQ13-Expectativas-de-Mercado.pdf), and [IBGE IPCA index](https://sidra.ibge.gov.br/tabela/1737).

The realized rate for a window from month M through M+H−1 is `100 × (IBGE index at M+H−1 / IBGE index at M−1 − 1)`. Signed error is forecast minus realized inflation, in percentage points.

## 1. Load the validated data

The IBGE monthly index CSV is generated locally using `projects/download_ibge_ipca.R`. The Focus SQLite snapshot is local; `dados/atual.json` identifies it. Run from the repository root or from `projects/`.

In [ ]:
suppressPackageStartupMessages({
  library(DBI)
  library(RSQLite)
  library(jsonlite)
})

project_root <- if (file.exists('dados/atual.json')) {
  '.'
} else if (file.exists('../dados/atual.json')) {
  '..'
} else {
  stop('Cannot find dados/atual.json; run from the repository root or projects/.')
}
snapshot <- fromJSON(file.path(project_root, 'dados/atual.json'))
ibge <- read.csv(file.path(project_root, 'projects/data/ibge_ipca_monthly_index_200110_202512.csv'),
                 colClasses = c('character', 'numeric'))
expected_index_months <- format(seq(as.Date('2001-10-01'), as.Date('2025-12-01'), by = 'month'), '%Y%m')
stopifnot(identical(ibge$month, expected_index_months),
          all(is.finite(ibge$ipca_index)), all(ibge$ipca_index > 0))
cat('Focus snapshot:', snapshot$coleta, '| IBGE index months:', nrow(ibge), '\n')

## 2. Match monthly forecasts to realized windows

Select the last Focus date of each month. At each date, require 12 monthly IPCA medians for the current month through month 11. The constructed 3-, 6-, and 12-month forecasts use cumulative products of the first 3, 6, and 12 medians. The same IBGE index provides realized inflation over each corresponding window.

In [ ]:
con <- dbConnect(SQLite(), file.path(project_root, snapshot$banco))
monthly <- tryCatch(dbGetQuery(con, '
  WITH anchors AS (
    SELECT substr(Data, 1, 7) AS forecast_month, max(Data) AS forecast_date
    FROM ExpectativaMercadoMensais
    WHERE Indicador = ? AND baseCalculo = ? AND Data >= ? AND Data < ?
    GROUP BY substr(Data, 1, 7)
  ), aligned AS (
    SELECT a.forecast_month, a.forecast_date,
           m.DataReferencia AS target_month, m.Mediana AS monthly_forecast_pct,
           (CAST(substr(m.DataReferencia, 4, 4) AS INTEGER)
             - CAST(substr(a.forecast_month, 1, 4) AS INTEGER)) * 12
             + CAST(substr(m.DataReferencia, 1, 2) AS INTEGER)
             - CAST(substr(a.forecast_month, 6, 2) AS INTEGER) AS months_ahead
    FROM anchors a CROSS JOIN ExpectativaMercadoMensais m
    WHERE m.Data = a.forecast_date AND m.Indicador = ? AND m.baseCalculo = ?
  )
  SELECT forecast_month, forecast_date, target_month,
         monthly_forecast_pct, months_ahead
  FROM aligned WHERE months_ahead BETWEEN 0 AND 11
  ORDER BY forecast_month, months_ahead
', params = list('IPCA', 0L, '2001-11-01', '2025-02-01', 'IPCA', 0L)),
finally = dbDisconnect(con))

parts <- split(monthly, monthly$forecast_month)
expected_forecast_months <- format(seq(as.Date('2001-11-01'), as.Date('2025-01-01'), by = 'month'), '%Y-%m')
stopifnot(identical(names(parts), expected_forecast_months),
          all(vapply(parts, nrow, integer(1)) == 12L))

panel <- do.call(rbind, lapply(parts, function(part) {
  if (anyNA(part$monthly_forecast_pct) || !all(part$months_ahead == 0:11)) {
    stop('Incomplete monthly forecast at ', part$forecast_month[1])
  }
  month <- part$forecast_month[1]
  first_day <- as.Date(paste0(month, '-01'))
  lag_days <- as.integer(seq(first_day, by = 'month', length.out = 2)[2] - 1
                         - as.Date(part$forecast_date[1]))
  if (lag_days < 0L || lag_days > 7L) stop('Stale month-end forecast at ', month)
  end_ref <- part$target_month[c(3L, 6L, 12L)]
  end_months <- paste0(substr(end_ref, 4, 7), substr(end_ref, 1, 2))
  start_row <- match(format(first_day - 1, '%Y%m'), ibge$month)
  end_rows <- match(end_months, ibge$month)
  if (is.na(start_row) || anyNA(end_rows)) stop('Missing IBGE index at ', month)
  forecast <- 100 * (cumprod(1 + part$monthly_forecast_pct / 100)[c(3L, 6L, 12L)] - 1)
  realized <- 100 * (ibge$ipca_index[end_rows] / ibge$ipca_index[start_row] - 1)
  data.frame(forecast_month = month, forecast_date = part$forecast_date[1],
             days_before_month_end = lag_days,
             forecast_3m_pct = forecast[1], realized_3m_pct = realized[1],
             forecast_6m_pct = forecast[2], realized_6m_pct = realized[2],
             forecast_12m_pct = forecast[3], realized_12m_pct = realized[3])
}))
row.names(panel) <- NULL
stopifnot(nrow(panel) == 279L)
for (h in c(3L, 6L, 12L)) {
  panel[[paste0('error_', h, 'm_pp')]] <- panel[[paste0('forecast_', h, 'm_pct')]] - panel[[paste0('realized_', h, 'm_pct')]]
  panel[[paste0('absolute_error_', h, 'm_pp')]] <- abs(panel[[paste0('error_', h, 'm_pp')]])
}
cat('Matched forecast months:', nrow(panel), '| monthly medians used:', nrow(monthly), '\n')
head(panel, 3)

## 3. Add the BCB's published 12-month benchmark

The official `Suavizada = 'N'` median and its `DesvioPadrao` are matched to the same month-end dates. Comparing official and constructed 12-month medians measures the aggregation difference; the published dispersion is used only with the published 12-month forecast error.

In [ ]:
con <- dbConnect(SQLite(), file.path(project_root, snapshot$banco))
official_all <- tryCatch(dbGetQuery(con, '
  SELECT substr(Data, 1, 7) AS forecast_month, Data AS forecast_date,
         Mediana AS published_12m_pct, DesvioPadrao AS disagreement_12m_pp
  FROM ExpectativasMercadoInflacao12Meses
  WHERE Indicador = ? AND Suavizada = ? AND baseCalculo = ?
    AND Data >= ? AND Data < ?
  ORDER BY Data
', params = list('IPCA', 'N', 0L, '2001-11-01', '2025-02-01')),
finally = dbDisconnect(con))
official <- official_all[!duplicated(official_all$forecast_month, fromLast = TRUE), ]
stopifnot(identical(official$forecast_month, panel$forecast_month),
          identical(official$forecast_date, panel$forecast_date),
          !anyNA(official$published_12m_pct), !anyNA(official$disagreement_12m_pp))
panel$published_12m_pct <- official$published_12m_pct
panel$disagreement_12m_pp <- official$disagreement_12m_pp
panel$published_error_12m_pp <- panel$published_12m_pct - panel$realized_12m_pct
panel$published_absolute_error_12m_pp <- abs(panel$published_error_12m_pp)
panel$constructed_minus_published_12m_pp <- panel$forecast_12m_pct - panel$published_12m_pct
panel$focus_snapshot <- snapshot$coleta
write.csv(panel, file.path(project_root, 'projects/data/ipca_rolling_horizon_comparison_200111_202501.csv'), row.names = FALSE)
cat('Saved matched panel with', nrow(panel), 'rows.\n')

## 4. Compare accuracy across horizons

Mean absolute error (MAE) is reported in percentage points. Because longer windows accumulate more monthly inflation, the table also shows MAE divided by the number of months as a descriptive scale check. This ratio is not a separate measure of forecaster skill.

In [ ]:
horizons <- c(3L, 6L, 12L)
accuracy <- do.call(rbind, lapply(horizons, function(h) {
  error <- panel[[paste0('error_', h, 'm_pp')]]
  absolute <- abs(error)
  data.frame(horizon_months = h, n = length(error),
             mean_signed_error_pp = mean(error),
             mae_pp = mean(absolute), mae_per_month_pp = mean(absolute) / h,
             median_absolute_error_pp = median(absolute),
             p90_absolute_error_pp = unname(quantile(absolute, 0.90)))
}))
print(accuracy, row.names = FALSE, digits = 4)

options(repr.plot.width = 10, repr.plot.height = 7)
par(bg = 'white')
plot(ecdf(panel$absolute_error_3m_pp), do.points = FALSE, verticals = TRUE,
     xlim = c(0, max(panel$absolute_error_12m_pp)), col = 'steelblue4', lwd = 2,
     xlab = 'Absolute forecast error (p.p.)', ylab = 'Empirical cumulative probability',
     main = 'Error distributions by rolling forecast horizon')
lines(ecdf(panel$absolute_error_6m_pp), do.points = FALSE, verticals = TRUE,
      col = 'darkorange2', lwd = 2)
lines(ecdf(panel$absolute_error_12m_pp), do.points = FALSE, verticals = TRUE,
      col = 'darkgreen', lwd = 2)
legend('bottomright', c('3 months', '6 months', '12 months'),
       col = c('steelblue4', 'darkorange2', 'darkgreen'), lty = 1, lwd = 2, bty = 'n')

**Validated snapshot results (279 matched months):**

| Horizon | Mean signed error (p.p.) | MAE (p.p.) | MAE per month (p.p.) | 90th percentile absolute error (p.p.) |
| --- | ---: | ---: | ---: | ---: |
| 3 months | -0.165 | 0.434 | 0.145 | 0.912 |
| 6 months | -0.436 | 0.953 | 0.159 | 2.061 |
| 12 months, constructed | -1.034 | 1.794 | 0.150 | 3.764 |

![Empirical CDFs of absolute forecast errors at 3, 6, and 12 months](figures/ipca_rolling_horizon_ecdf.png)



## 5. Compare the constructed and published 12-month forecasts

The two medians need not match: the BCB compounds each institution's monthly forecasts and then takes the median, while the constructed series compounds monthly medians calculated separately. This check quantifies that difference in the observed sample.

In [ ]:
gap <- panel$constructed_minus_published_12m_pp
aggregation_gap <- data.frame(
  months = length(gap), mean_gap_pp = mean(gap),
  mean_absolute_gap_pp = mean(abs(gap)),
  p90_absolute_gap_pp = unname(quantile(abs(gap), 0.90)),
  correlation = cor(panel$forecast_12m_pct, panel$published_12m_pct)
)
print(aggregation_gap, row.names = FALSE, digits = 4)
par(bg = 'white')
plot(panel$published_12m_pct, panel$forecast_12m_pct, pch = 16,
     col = rgb(0.1, 0.3, 0.6, 0.45),
     xlab = 'Published BCB 12-month median (%)',
     ylab = 'Constructed 12-month forecast (%)',
     main = 'Published versus constructed 12-month forecasts')
abline(0, 1, col = 'red', lty = 2)

The mean absolute gap between constructed and published 12-month forecasts is **0.049 p.p.**; their correlation is **0.999**. The published 12-month forecast has an MAE of **1.809 p.p.**, versus **1.794 p.p.** for the constructed series. This is a validation check, not proof that the two series are interchangeable.

![Published versus constructed 12-month forecasts](figures/ipca_official_vs_constructed_12m.png)



## 6. Is published forecast disagreement associated with error?

`DesvioPadrao` measures dispersion across the BCB's published 12-month forecasts. It is **not** a calibrated predictive standard deviation for future IPCA. Spearman correlation describes association with the published forecast's eventual absolute error; it does not establish causation.

In [ ]:
rho <- cor(panel$disagreement_12m_pp, panel$published_absolute_error_12m_pp,
           method = 'spearman')
cat(sprintf('Spearman rho = %.3f\n', rho))
par(bg = 'white')
plot(panel$disagreement_12m_pp, panel$published_absolute_error_12m_pp,
     pch = 16, col = rgb(0.1, 0.3, 0.6, 0.4),
     xlab = 'Published 12-month forecast dispersion (p.p.)',
     ylab = 'Absolute published 12-month error (p.p.)',
     main = 'Disagreement and subsequent forecast error')
legend('topright', sprintf('Spearman rho = %.2f', rho), bty = 'n')

The observed Spearman correlation is **0.245**. The moving-block bootstrap below quantifies its uncertainty.

![Published 12-month forecast disagreement versus subsequent absolute error](figures/ipca_disagreement_vs_error_12m.png)



## 7. Simulate uncertainty with a paired block bootstrap

Each draw samples **12-month circular blocks** of the monthly *vector* of 3-, 6-, and 12-month absolute errors, along with published 12-month disagreement and error. This preserves same-month pairing and some serial dependence from overlapping outcome windows. Ten thousand draws provide percentile intervals for MAE differences and the disagreement correlation. These intervals depend on the block-bootstrap approximation and do not account for structural changes in the economy.

In [ ]:
set.seed(20261102)
B <- 10000L
block_length <- 12L
n <- nrow(panel)
absolute_errors <- as.matrix(panel[c('absolute_error_3m_pp',
                                     'absolute_error_6m_pp',
                                     'absolute_error_12m_pp')])
boot_mae <- matrix(NA_real_, nrow = B, ncol = 3L)
boot_rho <- numeric(B)
for (b in seq_len(B)) {
  starts <- sample.int(n, ceiling(n / block_length), replace = TRUE)
  idx <- ((rep(starts - 1L, each = block_length)
           + rep(0:(block_length - 1L), times = length(starts))) %% n) + 1L
  idx <- idx[seq_len(n)]
  boot_mae[b, ] <- colMeans(absolute_errors[idx, , drop = FALSE])
  boot_rho[b] <- suppressWarnings(cor(panel$disagreement_12m_pp[idx],
                                       panel$published_absolute_error_12m_pp[idx],
                                       method = 'spearman'))
}
stopifnot(all(is.finite(boot_mae)), all(is.finite(boot_rho)))
boot_diff <- cbind('6 minus 3' = boot_mae[, 2] - boot_mae[, 1],
                   '12 minus 6' = boot_mae[, 3] - boot_mae[, 2],
                   '12 minus 3' = boot_mae[, 3] - boot_mae[, 1])
observed_mae <- colMeans(absolute_errors)
observed_diff <- c(observed_mae[2] - observed_mae[1],
                   observed_mae[3] - observed_mae[2],
                   observed_mae[3] - observed_mae[1])
limits <- apply(boot_diff, 2, quantile, probs = c(0.025, 0.975))
comparison <- data.frame(contrast = colnames(boot_diff),
                         observed_difference_pp = observed_diff,
                         lower_95_pp = limits[1, ], upper_95_pp = limits[2, ],
                         fraction_positive = colMeans(boot_diff > 0))
print(comparison, row.names = FALSE, digits = 4)
rho_limits <- quantile(boot_rho, c(0.025, 0.975))
print(data.frame(spearman_rho = rho,
                 lower_95 = unname(rho_limits[1]), upper_95 = unname(rho_limits[2])),
      row.names = FALSE, digits = 4)
checkpoints <- c(1000L, 5000L, 10000L)
convergence <- do.call(rbind, lapply(checkpoints, function(k) {
  difference <- boot_diff[seq_len(k), '12 minus 3']
  data.frame(draws = k, mean_12_minus_3_pp = mean(difference),
             lower_95_pp = unname(quantile(difference, 0.025)),
             upper_95_pp = unname(quantile(difference, 0.975)),
             rho_lower_95 = unname(quantile(boot_rho[seq_len(k)], 0.025)),
             rho_upper_95 = unname(quantile(boot_rho[seq_len(k)], 0.975)))
}))
print(convergence, row.names = FALSE, digits = 4)

### Block-length sensitivity

Repeat the two headline intervals with 6- and 24-month blocks (5,000 draws each). This tests whether the conclusions are an artifact of the 12-month block choice. The intervals remain approximate because macroeconomic regimes can persist longer than any of these blocks.


In [ ]:
set.seed(20261103)
block_sensitivity <- do.call(rbind, lapply(c(6L, 24L), function(block_length) {
  draws <- 5000L
  diff_draws <- numeric(draws)
  rho_draws <- numeric(draws)
  for (b in seq_len(draws)) {
    starts <- sample.int(n, ceiling(n / block_length), replace = TRUE)
    idx <- ((rep(starts - 1L, each = block_length)
             + rep(0:(block_length - 1L), times = length(starts))) %% n) + 1L
    idx <- idx[seq_len(n)]
    diff_draws[b] <- mean(panel$absolute_error_12m_pp[idx] - panel$absolute_error_3m_pp[idx])
    rho_draws[b] <- suppressWarnings(cor(panel$disagreement_12m_pp[idx],
                                         panel$published_absolute_error_12m_pp[idx],
                                         method = 'spearman'))
  }
  data.frame(block_months = block_length,
             diff_lower_95_pp = unname(quantile(diff_draws, 0.025)),
             diff_upper_95_pp = unname(quantile(diff_draws, 0.975)),
             rho_lower_95 = unname(quantile(rho_draws, 0.025)),
             rho_upper_95 = unname(quantile(rho_draws, 0.975)))
}))
print(block_sensitivity, row.names = FALSE, digits = 4)


With 6-month blocks, the 95% interval for the 12-minus-3-month MAE difference is **0.853–1.958 p.p.** and the interval for the disagreement correlation is **0.062–0.421**. With 24-month blocks, the intervals are **0.731–2.148 p.p.** and **0.014–0.415**. The positive association is weaker than the horizon contrast and remains sensitive to the historical sample.


## 8. Inspect the bootstrap distribution

The solid line is the observed difference in MAE between 12 and 3 months; dotted lines mark the bootstrap percentile interval. The red line marks zero difference.

In [ ]:
par(bg = 'white')
hist(boot_diff[, '12 minus 3'], breaks = 35, col = 'lightblue', border = 'white',
     main = 'Block-bootstrap MAE difference: 12 minus 3 months',
     xlab = 'Difference in MAE (p.p.)')
abline(v = observed_diff[3], col = 'navy', lwd = 2)
abline(v = limits[, 3], col = 'navy', lty = 3, lwd = 2)
abline(v = 0, col = 'red', lty = 2, lwd = 2)

With 10,000 draws and 12-month blocks, the observed 12-minus-3-month MAE difference is **1.360 p.p.** (percentile 95% interval: **0.788 to 2.089 p.p.**). The published 12-month disagreement–error Spearman correlation is **0.245** (percentile 95% interval: **0.046 to 0.413**).

![Bootstrap distribution of the 12-minus-3-month MAE difference](figures/ipca_rolling_horizon_bootstrap_difference.png)



### Interpretation limits

The cumulative outcomes have different scales, and adjacent windows overlap. Lower absolute error at a shorter horizon does not by itself prove better forecaster skill. The published 12-month dispersion can be evaluated only with the published 12-month forecast; aggregate monthly dispersions do not identify a comparable 3- or 6-month cross-sectional dispersion. The block bootstrap captures some dependence but cannot make this historical sample representative of every macroeconomic regime.